# 14주차 — LangGraph (3) 멀티 에이전트 · 스트리밍 · 서비스화 (Colab판)

「최신인공지능」 2026 · 14주차 실습

| 실습 | 교시 | 내용 | Colab |
|------|------|------|---|
| 실습 1 ★★ | 1교시 | **Supervisor 패턴** — 역할이 다른 에이전트를 관리자가 배분 | ✅ |
| 실습 2 | 2교시 | **스트리밍** — 진행 상황을 흘려보낸다 | ✅ |
| **실습 3** ★★ | 2교시 | `langgraph dev` + **LangGraph Studio** | ❌ **실습실 PC 전용** |
| 1절 | 3교시 | 운영 관점의 LangSmith — 모니터링 · 피드백 | ✅ |

> ### ❌ 실습 3은 이 노트북으로 대체되지 않습니다
>
> `langgraph dev` 는 **로컬 서버를 띄우고 브라우저가 그 서버에 붙는** 방식입니다.
> Colab 런타임의 `localhost` 에는 **브라우저가 접근할 수 없습니다.**
> 터널(ngrok 등)로 우회해도 CORS·인증이 걸려 **18분 실습에 부적합**합니다.
>
> **실습 3은 반드시 실습실 PC 에서 진행하십시오.**
> 아래에 `langgraph.json` 과 실행 절차를 정리해 두었습니다 (실행은 실습실에서).
>
> 🔶 **전날 반드시 실습실 PC 에서 `langgraph dev` 를 1회 띄워 보십시오.**
> 안 되면 **교수 시연으로 전환**하십시오 — 20분을 통째로 날리지 마십시오. ★

## 0. 환경 준비

In [ ]:
# ══════════════════════════════════════════════════════════════
#  Colab 환경 준비 — 매 세션 1회 실행 (재실행 안전)
# ══════════════════════════════════════════════════════════════
TOOL_MODEL_NAME = "qwen3:4b"       # 🔶 9주차에서 확정한 도구 호출 모델

WEEK_MODELS   = ["tool"]
WEEK_PACKAGES = ("langchain langchain-core langchain-ollama python-dotenv "
                 "pydantic langsmith langgraph grandalf")
WEEK_SECRETS  = ["LANGSMITH_API_KEY"]

# ──────────────────────────────────────────────────────────────
import os, shutil, subprocess, sys, time, urllib.request

IN_COLAB = "google.colab" in sys.modules
def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

GPU  = shutil.which("nvidia-smi") is not None and sh("nvidia-smi").returncode == 0
CHAT = os.environ.setdefault("MODEL",      "gemma3:4b" if GPU else "gemma3:1b")
TOOL = os.environ.setdefault("TOOL_MODEL", TOOL_MODEL_NAME)

print(f"[1/5] 런타임   {'GPU 있음 ✅' if GPU else 'CPU 전용 ⚠️'}   →  도구 모델 {TOOL}")
if not GPU:
    print("       ⚠️⚠️ 멀티 에이전트는 호출이 곱으로 늘어납니다. T4 GPU 가 사실상 필수입니다.")

print("[2/5] 패키지 설치 중…")
r = sh(f"{sys.executable} -m pip install -q {WEEK_PACKAGES}")
print("       ✅ 완료" if r.returncode == 0 else "       ❌ 실패\n" + r.stderr[-600:])

if shutil.which("ollama") is None:
    print("[3/5] Ollama 설치 중… (약 30초)")
    sh("curl -fsSL https://ollama.com/install.sh | sh")
print("[3/5] Ollama  " + ("✅ 준비됨" if shutil.which("ollama") else "❌ 설치 실패"))

def alive():
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        return True
    except Exception:
        return False

if not alive():
    subprocess.Popen(["ollama", "serve"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(60):
        if alive():
            break
        time.sleep(1)
print("[4/5] 서버    " + ("✅ 응답함" if alive() else "❌ 미응답 — 이 셀을 다시 실행하세요"))

have = {ln.split()[0] for ln in sh("ollama list").stdout.splitlines()[1:] if ln.strip()}
if TOOL in have:
    print(f"[5/5] {TOOL:<20s} ✅ 이미 있음")
else:
    print(f"[5/5] {TOOL:<20s} ⏳ 내려받는 중…")
    t0 = time.time()
    r = sh(f"ollama pull {TOOL}")
    print(f"       {'✅ 완료' if r.returncode == 0 else '❌ 실패'}  ({time.time() - t0:.0f}초)")

for k in WEEK_SECRETS:
    if not os.getenv(k) and IN_COLAB:
        try:
            from google.colab import userdata
            os.environ[k] = userdata.get(k)
        except Exception:
            pass
    print(f"[키]  {k:<20s} " + ("✅ 설정됨" if os.getenv(k) else "⬜ 없음"))

os.environ.setdefault("LANGSMITH_PROJECT", "week14-multiagent")
os.environ.setdefault("LANGSMITH_TRACING", "true")

print("\n" + "=" * 62)
print(f"준비 완료 — TOOL_MODEL='{TOOL}'")
print("=" * 62)

## 실습 1 ★★ (1교시) — Supervisor 패턴

### 왜 하나로는 안 되는가

```
도구 2~3개   →  잘 고른다 ✅
도구 5개     →  가끔 헷갈린다
도구 10개    →  ⚠️ 못 고른다        ← 9주차에서 이미 확인한 현상
```

| 실패 | 원인 |
|---|---|
| 엉뚱한 도구 선택 | 설명문 10개를 다 읽고 비교하기 어려움 |
| 도구를 아예 안 씀 | 선택지가 많아 판단을 회피 |
| **프롬프트 충돌** ★ | "간결히 답하라" + "근거를 상세히 인용하라" + … 지시가 부딪힘 |

> ⚠️ **9주차의 대응은 "도구 수를 2~3개로 제한"** 이었습니다.
> 그런데 **정말 10개가 필요한 서비스**라면? — **제한이 답이 될 수 없습니다.**

### 사람은 어떻게 하는가 → **역할별로 나누고, 관리자가 일을 배분한다** ★

```
              ┌──────────────┐
  질문 ──────▶│  Supervisor  │◀──────┐
              └──────┬───────┘       │ 결과를 보고 다음 담당자 결정
        ┌────────────┼────────────┐  │
        ▼            ▼            ▼  │
 ┌───────────┐ ┌──────────┐ ┌─────────┴─┐
 │ 검색 에이전트│ │계산 에이전트│ │작성 에이전트│
 │ (도구 1개)  │ │ (도구 1개) │ │ (도구 0개) │
 └───────────┘ └──────────┘ └───────────┘
```

> ★ **구조적으로는 12주차 조건부 엣지 + 순환입니다.**
> Supervisor 가 라우팅 노드이고, 워커가 끝나면 다시 Supervisor 로 돌아옵니다.
> **새 개념이 아니라 조합입니다.**
>
> ★ 개별 에이전트는 **9주차 원칙(도구 2~3개)을 그대로 지킵니다.**
> 각자 적은 도구 + 짧고 명확한 지시. 전체로는 3개를 씁니다.

> ### ⚠️⚠️ 비용은 **곱으로** 늘어납니다
>
> Supervisor 판단 1회 + 워커 실행 → 다시 Supervisor …
> 에이전트 3개 × 순환 = **단일 에이전트의 몇 배 호출** ⚠️
> → 로컬 모델 기준 **에이전트 2~3개, `recursion_limit` 은 낮게.**

In [ ]:
from typing import Annotated, Literal, TypedDict

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from pydantic import BaseModel, Field

TOOL_MODEL      = os.environ["TOOL_MODEL"]
MAX_STEPS       = 6      # ★ 종료 안전장치
RECURSION_LIMIT = 15

llm = ChatOllama(model=TOOL_MODEL, temperature=0)


# ── 도구: 역할별로 '적게' 준다 ★ ─────────────────────────────
@tool
def multiply(a: int, b: int) -> int:
    """두 정수를 곱한다. 정확한 곱셈이 필요할 때 사용한다."""
    return a * b


@tool
def search_docs(query: str) -> str:
    """사내·학내 문서에서 관련 내용을 찾는다. 규정·학칙 확인이 필요할 때 사용한다."""
    # 🔶 11주차 retriever 를 연결하면 실제 검색이 됩니다 ★
    #    docs = STORE.as_retriever(search_kwargs={"k": 3}).invoke(query)
    #    return "\n".join(d.page_content for d in docs)
    return f"[검색결과] '{query}' 관련 규정: 일반휴학은 통산 6개 학기까지."


class State(TypedDict):
    messages: Annotated[list, add_messages]
    next: str
    steps: int


WORKERS = ("researcher", "calculator", "writer")


# ── ① Supervisor — 다음 담당자를 고른다 ★★ ───────────────────
class Route(BaseModel):
    """다음 담당자."""

    next: Literal["researcher", "calculator", "writer", "FINISH"] = Field(
        description="문서 조회가 필요하면 researcher, 계산이면 calculator, "
                    "최종 정리가 필요하면 writer, 모두 끝났으면 FINISH")


def supervisor(state: State) -> dict:
    sys_msg = SystemMessage(
        "너는 팀 관리자다. 지금까지의 대화를 보고 다음에 일할 담당자를 고르라. "
        "이미 필요한 정보가 모였고 정리도 끝났으면 FINISH 를 고르라.")
    step = state.get("steps", 0) + 1
    try:
        nxt = llm.with_structured_output(Route).invoke(
            [sys_msg] + list(state["messages"])).next
    except Exception as e:     # 🔶 소형 모델 대비 — 헤매면 정리하고 끝냅니다
        print(f"  [Supervisor {step}] ⚠️ 판단 실패 ({type(e).__name__}) → writer")
        nxt = "writer"
    print(f"  [Supervisor {step}] → {nxt}")
    return {"next": nxt, "steps": step}


def route(state: State) -> str:
    # ★ 종료 조건 두 겹: Supervisor 의 FINISH + 단계 상한
    if state["next"] == "FINISH" or state["steps"] >= MAX_STEPS:
        return "FINISH"
    return state["next"] if state["next"] in WORKERS else "FINISH"


# ── ② 워커 — 각자 도구 1~2개만 ★ ─────────────────────────────
def make_worker(name: str, tools: list, instruction: str):
    """🔶 워커 내부는 간이 구현입니다 (내부 순환 1회).

    정식으로는 각 워커를 **서브그래프**(13주차 ReAct 그래프)로 만들어 노드에 넣습니다.
    **수업 시간(18분) 제약상 단순화**한 것입니다. ★

        research_graph = research_builder.compile()        # 그 자체로 완결된 에이전트
        main_builder.add_node("research", research_graph)  # ★ 노드로 넣는다
        # compile() 결과가 Runnable 이므로 노드가 될 수 있습니다
        # — 5주차 "체인도 부품이다" 가 그래프에서도 성립합니다 ★
    """
    bound    = llm.bind_tools(tools) if tools else llm
    registry = {t.name: t for t in tools}

    def worker(state: State) -> dict:
        history  = [SystemMessage(instruction)] + list(state["messages"])
        msg      = bound.invoke(history)
        produced = [msg]

        # 도구 호출이 있으면 여기서 처리하고, 결과를 넣어 한 번 더 부른다
        for call in (getattr(msg, "tool_calls", None) or []):
            obj = registry.get(call["name"])
            if obj is None:
                continue
            produced.append(obj.invoke(call))     # ← 실행 주체는 여전히 우리 코드 ★

        if len(produced) > 1:
            produced.append(bound.invoke(history + produced))

        text = (produced[-1].content or "").strip()
        print(f"  [{name}] {text[:70]}")

        # ★ 관리자가 읽을 수 있게 '누가 무엇을 했는지' 를 남깁니다.
        #   워커의 원본 메시지를 그대로 쌓으면 tool_calls 짝이 어긋날 수 있습니다. 🔶
        return {"messages": [AIMessage(content=f"[{name}] {text}", name=name)]}

    return worker


sb = StateGraph(State)
sb.add_node("supervisor", supervisor)
sb.add_node("researcher",
            make_worker("researcher", [search_docs],
                        "너는 문서 조사 담당이다. 검색 도구로 사실만 확인해 보고하라."))
sb.add_node("calculator",
            make_worker("calculator", [multiply],
                        "너는 계산 담당이다. 계산 도구로 정확히 계산해 보고하라."))
sb.add_node("writer",
            make_worker("writer", [],
                        "너는 작성 담당이다. 앞의 결과를 종합해 최종 답변을 작성하라."))

sb.add_edge(START, "supervisor")
sb.add_conditional_edges("supervisor", route, {
    "researcher": "researcher", "calculator": "calculator",
    "writer": "writer", "FINISH": END,
})
for w in WORKERS:
    sb.add_edge(w, "supervisor")       # ★ 끝나면 관리자에게 복귀 = 순환

graph = sb.compile()

QUESTION = "휴학 최대 학기 수를 확인하고, 그 값에 6을 곱한 결과를 알려줘."
INPUTS   = {"messages": [HumanMessage(QUESTION)], "steps": 0}
CONFIG   = {"recursion_limit": RECURSION_LIMIT}

print(f"모델: {TOOL_MODEL} / MAX_STEPS {MAX_STEPS} / recursion_limit {RECURSION_LIMIT}")
print(f"\nQ: {QUESTION}\n")

out = graph.invoke(INPUTS, CONFIG)

print("\n" + "=" * 60)
print("최종:", out["messages"][-1].content.strip()[:250])
print(f"\nSupervisor 판단 횟수: {out['steps']}회 / 누적 메시지 {len(out['messages'])}개")
print("=" * 60)

### 관찰 ★★ — 질문 하나에 두 담당자가 필요하도록 설계했습니다

```
[Supervisor 1] → researcher      (휴학 최대 학기 수를 조회)
[researcher] 일반휴학은 통산 6개 학기까지
[Supervisor 2] → calculator      (6 × 6 계산)
[calculator] 36
[Supervisor 3] → writer          (종합)
[writer] 휴학 최대 학기 수는 6이며, 6을 곱하면 36입니다.
[Supervisor 4] → FINISH
```

| 관찰 | 짚어줄 말 |
|---|---|
| Supervisor 가 **여러 번** | 배분 → 실행 → **결과를 보고 다시 판단** ★ |
| 각 워커의 도구가 1개뿐 | *"9주차 원칙을 각자 지키면서 전체는 3개를 씁니다"* ★★ |
| **호출 횟수** | 단일 에이전트보다 확실히 많다 ⚠️ |
| Supervisor 가 헤매면 | 소형 모델의 한계 → **`MAX_STEPS` 로 방어** |

> 📌 **LangSmith 추적을 꼭 열어 보십시오.**
> **Run 개수와 총 토큰**이 13주차 단일 에이전트보다 얼마나 늘었는지 **숫자로** 확인하십시오.

> ### ⚖️ 결론: 멀티 에이전트는 정확도를 비용으로 삽니다
>
> 7주차의 판단 틀(정확도 vs 토큰 vs 지연)이 여기서도 그대로 적용됩니다.
> **도구가 3개면 단일 에이전트가 낫습니다.** 나누는 것은 **역할이 정말 갈릴 때**입니다. ★

In [ ]:
# ── 그래프 구조를 그림으로 ★ ──
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())

## 실습 2 (2교시) — 스트리밍: 진행 상황을 흘려보낸다

에이전트는 오래 걸립니다.

```
[4주차 단일 호출]   질문 ─── 5초 ───▶ 답변
[14주차 에이전트]   질문 ─── 40초 ────────────────────▶ 답변  ⚠️
                         그동안 화면은 아무것도 없다 → 사용자는 **껐다고 판단**합니다
```

### 두 종류의 스트리밍 ★★

| 종류 | 무엇이 흐르나 | 사용자가 보는 것 |
|---|---|---|
| 토큰 단위 | 글자 | "휴학은 통산 6개..." (글자가 흘러나옴) — 4주차 |
| **상태 단위** ★ | **노드 실행 결과** | **"검색 중... 계산 중... 정리 중..."** ★★ |

> ### ★★ 상태 단위가 에이전트의 핵심입니다
>
> 토큰 단위만으로는 **도구를 실행하는 동안 여전히 침묵**합니다(생성이 아니므로).
> **"지금 무엇을 하고 있는가"** 를 보여주는 것이 상태 단위입니다.

| `stream_mode` | 나오는 것 | 쓰임 |
|---|---|---|
| `"values"` | **전체 상태** 스냅샷 | 디버깅 (13주차) |
| **`"updates"`** ★ | **어느 노드가 무엇을 바꿨는지** | **진행 표시** ★★ |
| `"messages"` | LLM **토큰** | 답변 글자 흘리기 |

> 🔶 모드 이름과 지원 여부는 버전에 따라 다릅니다. **사전 확인** 후 배포하십시오.

In [ ]:
import time

# ★ 노드 이름을 **사용자 말로 번역**합니다 — UX 의 실체
#   "researcher" 가 아니라 "문서를 찾는 중"
LABELS = {
    "supervisor": "담당자 배정 중",
    "researcher": "문서를 찾는 중",
    "calculator": "계산하는 중",
    "writer":     "답변을 정리하는 중",
}

print(f"Q: {QUESTION}\n")
print("── 상태 단위 (stream_mode='updates') ★★ ─────────")
t0 = time.perf_counter()
for chunk in graph.stream(INPUTS, CONFIG, stream_mode="updates"):
    for node in chunk:
        print(f"  ▸ [{time.perf_counter() - t0:5.1f}s] {LABELS.get(node, node)}...")
TOTAL = time.perf_counter() - t0
print(f"  ▸ [{TOTAL:5.1f}s] 완료")

In [ ]:
# ── 토큰 단위 — 답변 글자 흘리기 ──
#    ★ writer 노드의 토큰만 흘립니다.
#      중간 노드의 출력까지 흘리면 화면이 지저분해집니다.
print("── 토큰 단위 (stream_mode='messages') ────────────")
print("--- 최종 답변 ---")
try:
    for msg, meta in graph.stream(INPUTS, CONFIG, stream_mode="messages"):
        if meta.get("langgraph_node") == "writer":      # ★ 작성 노드의 토큰만
            print(msg.content, end="", flush=True)
    print()
except Exception as e:      # 🔶 모드 지원 여부가 버전마다 다릅니다
    print(f"\n🔶 messages 모드에서 오류 ({type(e).__name__}: {str(e)[:60]})")
    print("   버전에 따라 지원이 다릅니다. 상태 단위만으로도 이 절은 성립합니다. ★")

print(f"""
관찰 포인트 ★

  첫 진행 표시가 **1~2초 만에** 뜸        "40초 침묵이 사라졌습니다" ★
  총 소요 시간은 **그대로** ({TOTAL:.1f}s)   4주차 결론 그대로 — **체감만 개선** ★
  노드 이름을 **사용자 말로 번역**         "researcher" 가 아니라 "문서를 찾는 중"
                                          — **UX 의 실체** ★
  토큰 스트리밍을 **writer 로 한정**       중간 노드까지 흘리면 화면이 지저분해짐

📌 **총 시간은 못 줄입니다. 사용자가 기다릴 수 있게 만드는 것**이 목표입니다.

💡 비동기(astream) 는 **여러 사용자 요청을 동시에** 처리할 때 필요합니다.
   ⚠️ 본 교과목에서는 직접 구현하지 않습니다 — `langgraph dev` 가 내부에서 처리합니다.
""")

## 실습 3 ★★ (2교시) — `langgraph dev` + Studio 〔실습실 PC 전용〕

> ### ❌ Colab 에서는 실행할 수 없습니다
>
> `langgraph dev` 는 로컬 서버(기본 `:2024`)를 띄우고 **브라우저가 그 서버에 직접 붙습니다.**
> Colab 런타임의 `localhost` 는 브라우저에서 보이지 않습니다.
> **실습실 PC 에서 아래 절차대로 진행하십시오.**

### ① 설치

```bash
pip install "langgraph-cli[inmem]"
```

### ② `langgraph.json` — 프로젝트 루트에 둡니다

```json
{
  "dependencies": ["."],
  "graphs": {
    "supervisor": "./supervisor.py:graph",
    "hitl": "../../13week/code/hitl_agent.py:graph"
  },
  "env": ".env"
}
```

| 키 | 의미 |
|---|---|
| `dependencies` | 어떤 패키지를 쓸지 (`"."` = 현재 프로젝트) |
| `graphs` | **`파일경로:변수명`** — 그 변수가 `compile()` 된 그래프여야 합니다 ★ |
| `env` | 환경변수 파일 |

### ③ 실행

```bash
langgraph dev
#   → Studio 접속 주소가 출력됩니다. 그 주소를 브라우저에서 엽니다.
#   → 포트가 충돌하면:  langgraph dev --port 2025
```

### ④ Studio 에서 할 수 있는 것 ★

| 기능 | 13주차에 코드로 하던 것 |
|---|---|
| 그래프 구조를 **화면으로** 봄 | `draw_mermaid()` |
| 노드를 클릭해 **상태를 직접 수정** | 상태 딕셔너리를 손으로 고침 |
| **중간 지점부터 재실행** | `invoke(None, past.config)` — **Time Travel 을 화면에서** ★★ |
| 실행 이력(체크포인트) 목록 | `get_state_history()` |

> ### ⚠️ `langgraph dev` 는 개발용입니다
> 인증·확장성·안정성이 없습니다. **운영 배포용이 아닙니다.**
>
> 🔶🔶 **이 절이 2교시 최대의 위험 지점입니다.**
> CLI 설치 · 경로 · 포트 충돌 · 브라우저 정책 등 변수가 많습니다.
> **전날 실습실 PC 에서 반드시 1회 테스트**하고, 안 되면 **교수 시연으로 전환**하십시오. ★

In [ ]:
# ── 실습실에서 쓸 langgraph.json 을 만들어 둡니다 (Colab 에서는 실행만) ──
#    좌측 📁 파일 패널에서 내려받아 실습실 프로젝트 루트에 두십시오.
import json
from pathlib import Path

CONFIG_JSON = {
    "dependencies": ["."],
    "graphs": {
        "supervisor": "./supervisor.py:graph",
        "hitl": "../../13week/code/hitl_agent.py:graph",
    },
    "env": ".env",
}

Path("langgraph.json").write_text(
    json.dumps(CONFIG_JSON, ensure_ascii=False, indent=2), encoding="utf-8")

print("✅ langgraph.json 생성 — 좌측 📁 파일 패널에서 내려받으십시오.")
print(Path("langgraph.json").read_text(encoding="utf-8"))
print("""
⚠️ 이 파일은 **실습실 PC** 의 프로젝트 루트에 두고 `langgraph dev` 로 실행합니다.
   Colab 에서는 실행되지 않습니다. ★
""")

## 3교시 1절 — 운영 관점의 LangSmith

### 개발 중과 운영 중은 다릅니다 ★★

| | **개발 중** (6~7주차) | **운영 중** (오늘) |
|---|---|---|
| 누가 쓰나 | **개발자 본인** | **실제 사용자** ★ |
| 입력 | 내가 만든 테스트 케이스 | **예상 못 한 입력** ⚠️ |
| 목적 | 버그를 찾는다 / 개선을 잰다 | **지금 잘 돌고 있나** / 만족하나 |
| 보는 것 | 개별 Trace 를 뜯어봄 | **집계 지표**(오류율·지연·비용) ★ |
| 평가 | 내가 만든 데이터셋 | **실제 사용 로그에서 계속 확충** ★ |

```
개발:  "이 프롬프트가 나은가?"      → 데이터셋에 걸어 A/B
운영:  "어제보다 오류가 늘었나?"     → 대시보드 ★
       "어떤 질문에서 실패하나?"     → 실패 로그 → 데이터셋에 추가 ★★
```

### ① 모니터링 — 집계 지표 ★

| 지표 | 이상 신호 |
|---|---|
| **오류율** | 갑자기 상승 → 모델·API 장애, 입력 패턴 변화 |
| **지연 분포(P50/P95)** ★ | 평균은 괜찮은데 **P95 가 튀면** 일부 사용자가 매우 느림 ⚠️ |
| **토큰·비용** | 예상 밖 증가 → **순환이 과하게 돌고 있을 수 있음**(12·13주차) ★ |
| 호출량 | 급증 → 한도 소진 위험 (6주차 무료 한도) |

> ★ **평균이 아니라 분포를 보십시오.**
> "평균 3초" 인데 P95 가 40초면 **20명 중 1명은 40초를 기다립니다.**
> **그 사람은 서비스를 떠납니다.**

### ② 사용자 피드백 수집 ★

답변마다 `run_id` 를 받아두고, 사용자의 👍/👎 를 그 실행에 붙입니다.

| 왜 중요한가 | 설명 |
|---|---|
| **정답을 모르는 실운영** | 데이터셋의 기대 출력이 없다. **사용자만이 만족 여부를 안다** ★ |
| **데이터셋의 원천** | 👎가 붙은 실행 → **다음 데이터셋 예제** ★★ |
| 우선순위 | 어떤 유형에서 불만이 많은지 → 개선 순서 결정 |

### ③ 지속적 평가 — 회귀 검사 ★

```
프롬프트 수정 / 모델 교체 / 문서 갱신
          ▼
같은 데이터셋으로 재평가        ← 7주차 3교시 그대로 ★
          ▼
회귀가 없으면 반영 / 있으면 되돌림
```

> ★ 7주차에 배운 회귀 방지 루틴이 운영에서 **"매번 도는 절차"** 가 됩니다.
> 소프트웨어의 CI 와 같은 위치입니다.

In [ ]:
# ── 🔶 실제로 run 하나를 만들고 거기에 피드백을 붙여 봅니다 ──
#    ⚠️ LANGSMITH_API_KEY 와 LANGSMITH_TRACING=true 가 필요합니다.
if not os.getenv("LANGSMITH_API_KEY"):
    print("🔶 LANGSMITH_API_KEY 가 없습니다. 🔑 보안 비밀에 넣고 부트스트랩을 다시 실행하십시오.")
else:
    from langchain_core.tracers.context import collect_runs
    from langsmith import Client

    print("에이전트를 한 번 실행하고, 그 run 에 피드백을 붙입니다.\n")

    with collect_runs() as cb:          # ★ 방금 실행의 run_id 를 받아둔다
        fb_out = graph.invoke(INPUTS, CONFIG)
        run_id = cb.traced_runs[0].id if cb.traced_runs else None

    print("답변:", fb_out["messages"][-1].content.strip()[:120])
    print("run_id:", run_id)

    if run_id is None:
        print("🔶 run 을 수집하지 못했습니다. LANGSMITH_TRACING=true 인지 확인하십시오.")
    else:
        try:
            Client().create_feedback(run_id=run_id, key="user_rating", score=1,
                                     comment="수업 시연 — 사용자가 👍 를 눌렀다고 가정")
            print("""
✅ 피드백을 붙였습니다. LangSmith 화면에서 이 run 을 열어 확인하십시오. ★

   이 👍/👎 가 **다음 데이터셋의 원천**이 됩니다.
   실제 사용자는 **개발자가 상상 못 한 입력**을 넣습니다.
   그중 실패한 것을 계속 담아 **다시는 그 실패가 재발하지 않게** 합니다. ★★
""")
        except Exception as e:          # 🔶 API 형태 차이
            print(f"🔶 create_feedback 실패 ({type(e).__name__}: {str(e)[:70]})")
            print("   버전에 따라 인자 형태가 다를 수 있습니다. 사전 확인하십시오.")

### 배포 시 고려사항 ★

| 항목 | 왜 중요한가 | 배운 곳 |
|---|---|---|
| **상태 저장소** | 재시작해도 대화가 남아야 함 | 13주차 체크포인터 |
| **비용 상한** | 순환·멀티 에이전트는 호출 폭증 ⚠️ | 4주차 `max_tokens`, 12주차 `recursion_limit` |
| **보안** | 도구 실행 권한·주입 방어 | **9주차 · 13주차 HITL** ★ |
| **관측** | 운영 중 무슨 일이 일어나는가 | **6~7주차 → 오늘** |
| 개인정보 | 추적에 프롬프트가 전송됨 | 6주차 1교시 |

> ⚠️⚠️ 클라우드 배포는 **개념과 요금 구조만** 다룹니다. **가입을 유도하지 마십시오.**
> 학생이 개인 카드를 등록하는 일이 없어야 합니다. (6주차 LangSmith 와 같은 원칙) ★
>
> ⚠️ **개인정보 주의**: 운영 로그에는 **실제 사용자 입력**이 들어갑니다.
> 6주차 1교시의 보안 원칙이 운영에서 훨씬 중요해집니다. ★

### ★ "배포하면 끝" 이 아니라 "배포하면 시작"

```
[흔한 오해]   개발 → 배포 → 끝
[실제]        개발 → 배포 → 관측 → 실패 수집 → 데이터셋 확충 → 개선 → 재평가 → …
                        └──────────────── 계속 돈다 ★ ───────────────┘
```

> 📌 **이 순환이 6·7주차를 배운 진짜 이유입니다.**
> 도구를 배운 것이 아니라 **개선의 절차**를 배운 것입니다.

## 오늘 확인할 것

- [ ] Supervisor 가 **여러 번** 판단하며 담당자를 배분하는 것을 봤다 ★★
- [ ] 각 워커의 도구가 **1개뿐**인데 전체로는 3개를 쓰는 구조를 확인했다 ★★
- [ ] LangSmith 에서 **13주차 단일 에이전트 대비 Run 수·토큰 증가**를 숫자로 확인했다 ★
- [ ] `stream_mode="updates"` 로 **진행 표시**를 만들었다 ★★
- [ ] 노드 이름을 **사용자 말로 번역**했다 (UX 의 실체) ★
- [ ] `langgraph.json` 을 만들었다 (**실행은 실습실 PC 에서**) 🔶
- [ ] LangSmith 에 **피드백(👍)** 을 붙여 봤다 ★

> ### ⚠️ 실습실에서 따로 할 것
>
> **실습 3 — `langgraph dev` + Studio.** Colab 에서는 불가능합니다.
> `pip install "langgraph-cli[inmem]"` → `langgraph.json` 배치 → `langgraph dev`

### 오늘의 한 줄

> **멀티 에이전트는 정확도를 비용으로 삽니다.**
> 그리고 **배포하면 끝이 아니라 시작**입니다.